# SQL Database Project — Synthetic Retail Bank

## 0–1. Configuration and helpers
Seed, reference date, sizes, and small helper functions (weighted choice, log-normal money draws, pence handling).

In [ ]:
import math
import os
import random
import sqlite3
from collections import defaultdict
from datetime import date, datetime, time, timedelta

# -----------------------------------------------------------------------------
# 0. CONFIGURATION
# -----------------------------------------------------------------------------
# A fixed seed makes the "random" data reproducible: anyone re-running the
# script gets the identical database, which is important for marking/audit.
SEED = 42
random.seed(SEED)

DB_PATH = "bank.db"

# A fixed "today" (rather than date.today()) also keeps the output reproducible.
TODAY = date(2026, 10, 30)

N_CUSTOMERS = 1200        # >= 1000 rows required by the brief
MAX_TXNS_PER_ACCOUNT = 45 # keeps the file a manageable size


# -----------------------------------------------------------------------------
# 1. SMALL HELPERS
# -----------------------------------------------------------------------------


def wchoice(options, weights):
    """Weighted random choice of a single item."""
    return random.choices(options, weights=weights, k=1)[0]


def lognorm(median, sigma):
    """
    Log-normal draw with a given median. Money amounts (spending, income,
    deposits) are right-skewed in real life: many small, few very large.
    A log-normal reproduces that shape far better than a uniform draw.
    """
    return median * math.exp(random.gauss(0, sigma))


def pence(pounds):
    """Money is handled internally as INTEGER pence to avoid float drift."""
    return int(round(pounds * 100))


def clip(x, lo, hi):
    return max(lo, min(hi, x))


def rand_date(start: date, end: date) -> date:
    """Uniformly random date between start and end (inclusive)."""
    return start + timedelta(days=random.randint(0, (end - start).days))


def rand_activity_time() -> time:
    """Time of day peaking mid-afternoon (customers rarely transact at 3 am)."""
    hour = int(random.triangular(6, 23, 13))
    return time(min(hour, 23), random.randint(0, 59), random.randint(0, 59))


def rand_branch_time() -> time:
    """Branch opening hours, used for account-opening/closing events."""
    return time(random.randint(9, 16), random.randint(0, 59), random.randint(0, 59))


## 2. Schema (DDL)
Six tables with primary keys, foreign keys, two compound primary keys, `CHECK` constraints and indexes. Design justifications are in the comments.

In [ ]:
SCHEMA = """
PRAGMA foreign_keys = ON;

CREATE TABLE branches (
    branch_id    INTEGER PRIMARY KEY,
    branch_name  TEXT    NOT NULL UNIQUE,
    city         TEXT    NOT NULL,
    region       TEXT    NOT NULL,
    sort_code    TEXT    NOT NULL UNIQUE
                 CHECK (sort_code GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9]'),
    branch_type  TEXT    NOT NULL                          -- ORDINAL
                 CHECK (branch_type IN ('Express','Standard','Flagship')),
    opened_date  TEXT    NOT NULL                          -- INTERVAL (date)
                 CHECK (opened_date GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'),
    staff_count  INTEGER NOT NULL CHECK (staff_count >= 0) -- RATIO
);

CREATE TABLE customers (
    customer_id        INTEGER PRIMARY KEY,
    first_name         TEXT NOT NULL,
    last_name          TEXT NOT NULL,
    gender             TEXT NOT NULL                       -- NOMINAL
                       CHECK (gender IN ('Female','Male','Non-binary')),
    date_of_birth      TEXT NOT NULL,                      -- INTERVAL (date)
    email              TEXT NOT NULL UNIQUE,
    phone              TEXT NOT NULL,
    city               TEXT NOT NULL,                      -- NOMINAL
    employment_status  TEXT NOT NULL                       -- NOMINAL
                       CHECK (employment_status IN
                         ('Employed','Self-employed','Student','Unemployed','Retired')),
    education_level    TEXT NOT NULL                       -- ORDINAL
                       CHECK (education_level IN
                         ('No formal qualification','Secondary','Vocational',
                          'Bachelor','Master','Doctorate')),
    annual_income      REAL NOT NULL CHECK (annual_income >= 0),   -- RATIO
    credit_score       INTEGER NOT NULL                    -- INTERVAL
                       CHECK (credit_score BETWEEN 300 AND 850),
    credit_rating      TEXT NOT NULL                       -- ORDINAL
                       CHECK (credit_rating IN
                         ('Poor','Fair','Good','Very Good','Exceptional')),
    customer_since     TEXT NOT NULL,                      -- INTERVAL (date)
    CHECK (customer_since > date_of_birth)
);

CREATE TABLE accounts (
    account_id            INTEGER PRIMARY KEY,
    branch_id             INTEGER NOT NULL REFERENCES branches(branch_id),
    account_number        TEXT    NOT NULL UNIQUE
                          CHECK (account_number GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]'),
    account_type          TEXT    NOT NULL                 -- NOMINAL
                          CHECK (account_type IN
                            ('Current','Premium Current','Student','Savings')),
    status                TEXT    NOT NULL                 -- NOMINAL
                          CHECK (status IN ('Active','Dormant','Closed')),
    open_date             TEXT    NOT NULL,                -- INTERVAL (date)
    close_date            TEXT,                            -- NULL while open
    balance               REAL    NOT NULL,                -- RATIO
    overdraft_limit       REAL    NOT NULL CHECK (overdraft_limit >= 0),      -- RATIO
    annual_interest_rate  REAL    NOT NULL
                          CHECK (annual_interest_rate BETWEEN 0 AND 10),      -- RATIO (%)
    CHECK ((status = 'Closed') = (close_date IS NOT NULL)),
    CHECK (close_date IS NULL OR close_date > open_date),
    CHECK (balance >= -overdraft_limit)
);

-- Junction table (many-to-many) with a COMPOUND PRIMARY KEY.
CREATE TABLE account_holders (
    account_id   INTEGER NOT NULL REFERENCES accounts(account_id),
    customer_id  INTEGER NOT NULL REFERENCES customers(customer_id),
    holder_role  TEXT    NOT NULL CHECK (holder_role IN ('Primary','Joint')),
    added_date   TEXT    NOT NULL,
    PRIMARY KEY (account_id, customer_id)
);
-- Business rule: every account has exactly one 'Primary' holder.
CREATE UNIQUE INDEX uq_one_primary_holder
    ON account_holders(account_id) WHERE holder_role = 'Primary';

CREATE TABLE transactions (
    txn_id        INTEGER PRIMARY KEY,
    account_id    INTEGER NOT NULL REFERENCES accounts(account_id),
    txn_datetime  TEXT    NOT NULL,                        -- INTERVAL (datetime)
    txn_type      TEXT    NOT NULL                         -- NOMINAL
                  CHECK (txn_type IN
                    ('Salary','Pension','Deposit','Transfer In',
                     'Withdrawal','Card Payment','Direct Debit','Transfer Out','Fee')),
    direction     TEXT    NOT NULL CHECK (direction IN ('Credit','Debit')),
    category      TEXT    NOT NULL,                        -- NOMINAL
    channel       TEXT    NOT NULL                         -- NOMINAL
                  CHECK (channel IN
                    ('ATM','Branch','Mobile App','Online Banking',
                     'Card Terminal','Automated','System')),
    amount        REAL    NOT NULL CHECK (amount > 0),     -- RATIO
    balance_after REAL    NOT NULL,                        -- RATIO
    is_flagged    INTEGER NOT NULL DEFAULT 0 CHECK (is_flagged IN (0,1)),
    -- direction is derived from txn_type but stored for fast filtering;
    -- this CHECK guarantees the two can never contradict each other.
    CHECK ( (direction = 'Credit' AND txn_type IN ('Salary','Pension','Deposit','Transfer In'))
         OR (direction = 'Debit'  AND txn_type IN ('Withdrawal','Card Payment','Direct Debit','Transfer Out','Fee')) )
);

-- Compound primary key: one statement per account per month.
CREATE TABLE monthly_statements (
    account_id       INTEGER NOT NULL REFERENCES accounts(account_id),
    statement_month  TEXT    NOT NULL                      -- INTERVAL, 'YYYY-MM'
                     CHECK (statement_month GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]'),
    opening_balance  REAL    NOT NULL,                     -- RATIO
    closing_balance  REAL    NOT NULL,
    total_credits    REAL    NOT NULL CHECK (total_credits >= 0),
    total_debits     REAL    NOT NULL CHECK (total_debits >= 0),
    txn_count        INTEGER NOT NULL CHECK (txn_count > 0),
    lowest_balance   REAL    NOT NULL,
    PRIMARY KEY (account_id, statement_month)
);

-- Indexes on the columns most often used in JOINs / filters.
CREATE INDEX idx_txn_account_time ON transactions(account_id, txn_datetime);
CREATE INDEX idx_txn_type         ON transactions(txn_type);
CREATE INDEX idx_holders_customer ON account_holders(customer_id);
CREATE INDEX idx_accounts_branch  ON accounts(branch_id);
"""


## 3. Reference data

In [ ]:
FEMALE_NAMES = ["Olivia", "Amelia", "Isla", "Ava", "Emily", "Sophie", "Grace", "Mia",
                "Charlotte", "Ella", "Chloe", "Hannah", "Freya", "Lucy", "Ruby",
                "Priya", "Aisha", "Fatima", "Zara", "Maya", "Sarah", "Laura", "Emma",
                "Anna", "Katie", "Megan", "Holly", "Jessica", "Rebecca", "Nadia"]
MALE_NAMES = ["Oliver", "George", "Harry", "Jack", "Noah", "Leo", "Oscar", "Charlie",
              "Henry", "Thomas", "James", "William", "Daniel", "Samuel", "Joseph",
              "Arjun", "Omar", "Ali", "Rahul", "Mohammed", "Ryan", "Callum", "Liam",
              "Ethan", "Adam", "Ben", "Luke", "Aaron", "Hassan", "Sean"]
NEUTRAL_NAMES = ["Alex", "Jamie", "Taylor", "Morgan", "Charlie", "Robin", "Sam", "Jordan"]
SURNAMES = ["Smith", "Jones", "Taylor", "Brown", "Williams", "Wilson", "Johnson",
            "Davies", "Robinson", "Wright", "Thompson", "Evans", "Walker", "White",
            "Roberts", "Green", "Hall", "Wood", "Jackson", "Clarke", "Patel", "Khan",
            "Ahmed", "Hussain", "Singh", "Shah", "Begum", "Ali", "Lewis", "Harris",
            "Martin", "Cooper", "King", "Lee", "Baker", "Turner", "Parker", "Morgan",
            "Bell", "Murphy", "Kelly", "Campbell", "Stewart", "MacDonald", "Reid",
            "Scott", "Murray", "Graham", "Ross", "Fraser", "Hughes", "Edwards",
            "Griffiths", "Morris", "Price", "Rees", "Owen", "Phillips", "Mitchell",
            "Carter", "Ward", "Cook", "Bailey", "Hill", "Adams", "Moore", "Allen"]

# (city, region) for the 15 branch cities.
BRANCH_CITIES = [
    ("London", "London"), ("Manchester", "North West"), ("Birmingham", "West Midlands"),
    ("Leeds", "Yorkshire"), ("Glasgow", "Scotland"), ("Liverpool", "North West"),
    ("Bristol", "South West"), ("Edinburgh", "Scotland"), ("Sheffield", "Yorkshire"),
    ("Cardiff", "Wales"), ("Newcastle", "North East"), ("Nottingham", "East Midlands"),
    ("Belfast", "Northern Ireland"), ("Southampton", "South East"),
    ("Leicester", "East Midlands"),
]
FLAGSHIP_CITIES = {"London", "Manchester", "Birmingham", "Glasgow", "Edinburgh"}

# Customer home cities: weights roughly follow population. Some cities have no
# branch (customers then use their nearest/any branch), which is realistic.
CUSTOMER_CITY_WEIGHTS = {
    "London": 22, "Manchester": 8, "Birmingham": 8, "Leeds": 6, "Glasgow": 6,
    "Liverpool": 5, "Bristol": 5, "Edinburgh": 4, "Sheffield": 4, "Cardiff": 4,
    "Newcastle": 4, "Nottingham": 4, "Belfast": 3, "Southampton": 3, "Leicester": 3,
    "Norwich": 2, "Exeter": 1, "Aberdeen": 1, "Cambridge": 1, "York": 1,
    "Swansea": 1, "Plymouth": 1, "Brighton": 2, "Oxford": 1, "Inverness": 1,
}

EDUCATION_LEVELS = ["No formal qualification", "Secondary", "Vocational",
                    "Bachelor", "Master", "Doctorate"]          # ordered low -> high
EDUCATION_WEIGHTS = [6, 28, 17, 30, 15, 4]

# Spending behaviour: category -> (selection weight, median amount in GBP)
CARD_CATEGORIES = {
    "Groceries": (30, 42), "Transport": (12, 18), "Dining": (14, 28),
    "Shopping": (18, 55), "Entertainment": (8, 22), "Healthcare": (4, 35),
    "Travel": (4, 160), "Other": (10, 40),
}
DD_CATEGORIES = {   # direct debits (None = computed from income below)
    "Rent/Mortgage": (25, None), "Utilities": (25, 75), "Insurance": (12, 40),
    "Subscriptions": (20, 12), "Phone/Broadband": (18, 35),
}

CURRENT_TYPE_WEIGHTS = {"Card Payment": 36, "Direct Debit": 11, "Withdrawal": 9,
                        "Transfer Out": 7, "Fee": 1, "Salary": 13,
                        "Deposit": 6, "Transfer In": 7}
SAVINGS_TYPE_WEIGHTS = {"Deposit": 30, "Transfer In": 33, "Withdrawal": 12,
                        "Transfer Out": 25}

CREDIT_TYPES = {"Salary", "Pension", "Deposit", "Transfer In"}

# Average transactions per month by account type (drives how busy accounts are)
TXN_RATE_PER_MONTH = {"Current": 1.6, "Premium Current": 2.2, "Student": 1.4, "Savings": 0.35}
# Median opening deposit (GBP)
OPENING_MEDIAN = {"Current": 400, "Premium Current": 2500, "Student": 150, "Savings": 1000}

# Overdraft allowed by credit rating (GBP) - better credit => larger overdraft
OVERDRAFT_BY_RATING = {"Poor": 0, "Fair": 250, "Good": 500, "Very Good": 1000, "Exceptional": 2000}


## 4. Branches

In [ ]:
def generate_branches():
    rows = []
    used_codes = set()
    for i, (city, region) in enumerate(BRANCH_CITIES, start=1):
        # Fictional sort code, unique per branch
        while True:
            code = f"{random.randint(10, 99)}-{random.randint(10, 99)}-{random.randint(10, 99)}"
            if code not in used_codes:
                used_codes.add(code)
                break
        if city in FLAGSHIP_CITIES:
            btype, staff = "Flagship", random.randint(40, 80)
        elif random.random() < 0.7:
            btype, staff = "Standard", random.randint(15, 35)
        else:
            btype, staff = "Express", random.randint(5, 12)
        opened = rand_date(date(1985, 1, 1), date(2015, 12, 31))
        rows.append((i, f"{city} Central", city, region, code, btype,
                     opened.isoformat(), staff))
    return rows


## 5. Customers

In [ ]:



def pick_employment(age):
    """Employment status depends realistically on age."""
    if age < 23:
        return wchoice(["Student", "Employed", "Unemployed"], [55, 35, 10])
    if age < 65:
        return wchoice(["Employed", "Self-employed", "Unemployed", "Student"],
                       [72, 13, 12, 3])
    return wchoice(["Retired", "Employed", "Self-employed"], [85, 10, 5])


def pick_income(employment):
    """Annual income (ratio data: 0 really means 'no income')."""
    if employment == "Employed":
        x = lognorm(31000, 0.45)
        return int(round(clip(x, 14000, 250000), -2))
    if employment == "Self-employed":
        x = lognorm(33000, 0.65)
        return int(round(clip(x, 6000, 250000), -2))
    if employment == "Student":
        return int(round(random.uniform(0, 14000), -2))
    if employment == "Retired":
        x = lognorm(16000, 0.4)
        return int(round(clip(x, 8000, 80000), -2))
    # Unemployed: benefits / no income (some genuine zeros)
    return 0 if random.random() < 0.4 else int(round(random.uniform(3000, 9000), -2))


def credit_rating_from_score(score):
    """Standard FICO-style bands -> ORDINAL label derived from the INTERVAL score."""
    if score < 580:
        return "Poor"
    if score < 670:
        return "Fair"
    if score < 740:
        return "Good"
    if score < 800:
        return "Very Good"
    return "Exceptional"


def generate_customers():
    cities = list(CUSTOMER_CITY_WEIGHTS)
    weights = list(CUSTOMER_CITY_WEIGHTS.values())
    customers = []
    for cid in range(1, N_CUSTOMERS + 1):
        gender = wchoice(["Female", "Male", "Non-binary"], [49, 49, 2])
        first = (random.choice(FEMALE_NAMES) if gender == "Female"
                 else random.choice(MALE_NAMES) if gender == "Male"
                 else random.choice(NEUTRAL_NAMES))
        last = random.choice(SURNAMES)

        age = int(clip(random.gauss(44, 16), 19, 92))
        dob = TODAY - timedelta(days=int(age * 365.25) + random.randint(0, 364))
        employment = pick_employment(age)
        income = pick_income(employment)

        # Credit score correlates with income and age, plus random noise.
        score = (690 + 45 * math.log(max(income, 5000) / 30000)
                 + (age - 40) * 1.2 + random.gauss(0, 45))
        score = int(clip(round(score), 300, 850))

        # Became a customer after their 18th birthday and no earlier than 2000.
        earliest = max(dob + timedelta(days=18 * 365 + 5), date(2000, 1, 1))
        since = rand_date(earliest, TODAY - timedelta(days=30))

        customers.append({
            "id": cid, "first": first, "last": last, "gender": gender,
            "dob": dob, "age": age,
            # example.com is reserved for documentation => can never reach a real person
            "email": f"{first}.{last}{cid}@example.com".lower(),
            # 07700 900xxx is the range regulators reserve for fictional/drama use
            "phone": f"07700 900{random.randint(0, 999):03d}",
            "city": wchoice(cities, weights),
            "employment": employment,
            "education": wchoice(EDUCATION_LEVELS, EDUCATION_WEIGHTS),
            "income": income, "score": score,
            "rating": credit_rating_from_score(score),
            "since": since,
        })
    return customers


## 6. Accounts and joint holders

In [ ]:



def generate_account_specs(customers, branches):
    branch_by_city = {b[2]: b[0] for b in branches}
    all_branch_ids = [b[0] for b in branches]
    by_surname = defaultdict(list)
    for c in customers:
        by_surname[c["last"]].append(c)

    used_numbers = set()
    accounts, holders = [], []
    acc_id = 0

    for c in customers:
        n_acc = wchoice([1, 2, 3], [60, 30, 10])
        for k in range(n_acc):
            acc_id += 1

            # --- account type ---
            if k == 0:  # everyone's first account is a current-type account
                if c["employment"] == "Student" and c["age"] <= 26 and random.random() < 0.8:
                    atype = "Student"
                elif c["income"] > 55000 and random.random() < 0.35:
                    atype = "Premium Current"
                else:
                    atype = "Current"
            else:
                atype = "Savings" if random.random() < 0.75 else "Current"

            # --- dates ---
            open_d = c["since"] if k == 0 else rand_date(c["since"], TODAY - timedelta(days=10))

            # --- status ---
            status, close_d, last_activity = "Active", None, None
            roll = random.random()
            if roll < 0.10 and (TODAY - open_d).days >= 90:
                status = "Closed"
                close_d = rand_date(open_d + timedelta(days=30), TODAY - timedelta(days=1))
            elif roll < 0.15 and (TODAY - open_d).days >= 730:
                status = "Dormant"   # no activity for > 12 months
                last_activity = rand_date(open_d + timedelta(days=30),
                                          TODAY - timedelta(days=370))

            # --- branch: mostly the home-city branch ---
            if c["city"] in branch_by_city and random.random() < 0.85:
                branch_id = branch_by_city[c["city"]]
            else:
                branch_id = random.choice(all_branch_ids)

            # --- unique 8-digit account number ---
            while True:
                number = str(random.randint(10_000_000, 99_999_999))
                if number not in used_numbers:
                    used_numbers.add(number)
                    break

            # --- product terms ---
            if atype == "Savings":
                overdraft = 0
                rate = round(random.uniform(1.5, 4.5), 2)
            elif atype == "Student":
                overdraft = 1500
                rate = 0.0
            else:
                overdraft = OVERDRAFT_BY_RATING[c["rating"]] * (2 if atype == "Premium Current" else 1)
                rate = round(random.uniform(0.0, 0.5) if atype == "Current"
                             else random.uniform(0.5, 1.5), 2)

            accounts.append({
                "id": acc_id, "branch_id": branch_id, "number": number,
                "type": atype, "status": status, "open": open_d,
                "close": close_d, "last_activity": last_activity,
                "overdraft": overdraft, "rate": rate, "primary": c,
            })
            holders.append((acc_id, c["id"], "Primary", open_d.isoformat()))

            # --- optional joint holder (12% of non-student accounts) ---
            if atype != "Student" and random.random() < 0.12:
                for _ in range(10):  # a few tries to find a valid partner
                    if random.random() < 0.5 and len(by_surname[c["last"]]) > 1:
                        partner = random.choice(by_surname[c["last"]])  # often family
                    else:
                        partner = random.choice(customers)
                    if partner["id"] != c["id"] and partner["since"] <= open_d:
                        holders.append((acc_id, partner["id"], "Joint", open_d.isoformat()))
                        break
    return accounts, holders


## 7. Transaction simulation
Each account is simulated chronologically so balances always reconcile and never breach the overdraft limit.

In [ ]:



def build_txn(ttype, atype, income):
    """Return (category, channel, amount_in_pence) for a transaction type."""
    scale = clip((max(income, 8000) / 30000) ** 0.4, 0.6, 2.0)  # richer => spend more

    if ttype == "Salary":
        return "Income", "Automated", pence(income / 12 * 0.78 * random.uniform(0.98, 1.02))
    if ttype == "Pension":
        return "Income", "Automated", pence(income / 12 * 0.90 * random.uniform(0.98, 1.02))
    if ttype == "Deposit":
        median = 300 if atype == "Savings" else 200
        channel = wchoice(["Branch", "ATM", "Mobile App"], [50, 30, 20])
        return "Cash/Cheque Deposit", channel, pence(max(5, lognorm(median * scale, 0.8)))
    if ttype == "Transfer In":
        median = 400 if atype == "Savings" else 180
        channel = wchoice(["Mobile App", "Online Banking"], [55, 45])
        return "Transfer", channel, pence(max(5, lognorm(median * scale, 0.8)))
    if ttype == "Withdrawal":
        if atype == "Savings":
            channel = wchoice(["Branch", "Online Banking", "Mobile App"], [30, 35, 35])
            return "Savings Withdrawal", channel, pence(max(10, lognorm(250 * scale, 0.8)))
        amount = wchoice([10, 20, 30, 40, 50, 60, 80, 100, 150, 200],
                         [5, 25, 12, 10, 18, 5, 8, 10, 4, 3])  # ATMs dispense round sums
        return "Cash", wchoice(["ATM", "Branch"], [85, 15]), pence(amount)
    if ttype == "Card Payment":
        cat = wchoice(list(CARD_CATEGORIES), [v[0] for v in CARD_CATEGORIES.values()])
        median = CARD_CATEGORIES[cat][1] * scale
        channel = wchoice(["Card Terminal", "Online Banking"], [70, 30])
        return cat, channel, pence(max(1, lognorm(median, 0.7)))
    if ttype == "Direct Debit":
        cat = wchoice(list(DD_CATEGORIES), [v[0] for v in DD_CATEGORIES.values()])
        if cat == "Rent/Mortgage":
            median = clip(income / 12 * 0.28, 250, 2500)
            return cat, "Automated", pence(lognorm(median, 0.1))
        return cat, "Automated", pence(lognorm(DD_CATEGORIES[cat][1], 0.25))
    if ttype == "Transfer Out":
        channel = wchoice(["Mobile App", "Online Banking"], [55, 45])
        return "Transfer", channel, pence(max(5, lognorm(150 * scale, 0.8)))
    # Fee
    return "Bank Fees", "System", pence(random.choice([3.5, 5, 7.5, 10, 12]))


def simulate_account(acc):
    """
    Simulate one account's life. Returns a list of tuples:
    (datetime, acc_id, seq, txn_type, direction, category, channel,
     amount_p, balance_after_p, flagged)
    """
    atype = acc["type"]
    cust = acc["primary"]
    income, employment = cust["income"], cust["employment"]
    overdraft_p = pence(acc["overdraft"])
    open_d = acc["open"]

    # Last day on which ordinary activity may happen
    if acc["status"] == "Closed":
        end_d = acc["close"] - timedelta(days=1)
    elif acc["status"] == "Dormant":
        end_d = acc["last_activity"]
    else:
        end_d = TODAY

    rows, seq, balance = [], 0, 0

    def add(dt, ttype, category, channel, amount_p, flagged=0):
        nonlocal seq, balance
        direction = "Credit" if ttype in CREDIT_TYPES else "Debit"
        balance += amount_p if direction == "Credit" else -amount_p
        seq += 1
        rows.append((dt, acc["id"], seq, ttype, direction, category, channel,
                     amount_p, balance, flagged))

    # 1) Opening deposit
    opening = max(1000, pence(lognorm(OPENING_MEDIAN[atype], 0.6)))
    add(datetime.combine(open_d, rand_branch_time()), "Deposit", "Account Opening",
        wchoice(["Branch", "Online Banking"], [70, 30]), opening)

    # 2) Ordinary activity: volume scales with account age and product type
    total_days = max(1, (end_d - open_d).days)
    expected = TXN_RATE_PER_MONTH[atype] * total_days / 30.4
    n = int(clip(random.gauss(expected, 0.3 * expected), 3, MAX_TXNS_PER_ACCOUNT))
    stamps = sorted(datetime.combine(open_d + timedelta(days=random.randint(1, total_days)),
                                     rand_activity_time()) for _ in range(n))

    # Which transaction types suit this customer?
    if atype == "Savings":
        weights = dict(SAVINGS_TYPE_WEIGHTS)
    else:
        weights = dict(CURRENT_TYPE_WEIGHTS)
        if employment == "Employed":
            pass                                   # keeps 'Salary'
        elif employment == "Retired":
            weights["Pension"] = weights.pop("Salary")
        else:                                      # no payslip: irregular deposits instead
            weights["Deposit"] += weights.pop("Salary") * 0.6

    for dt in stamps:
        ttype = wchoice(list(weights), list(weights.values()))
        category, channel, amount_p = build_txn(ttype, atype, income)

        # Prevent breaching the overdraft: if the debit is unaffordable, the
        # customer pays money in instead (as a real person would have to).
        if ttype not in CREDIT_TYPES and amount_p > balance + overdraft_p:
            ttype = wchoice(["Deposit", "Transfer In"], [50, 50])
            category, channel, amount_p = build_txn(ttype, atype, income)

        flagged = 1 if (ttype in ("Card Payment", "Transfer Out") and random.random() < 0.006) else 0
        add(dt, ttype, category, channel, amount_p, flagged)

    # 3) Closure: customer withdraws any remaining money (or clears an overdraft)
    if acc["status"] == "Closed":
        dt = datetime.combine(acc["close"], time(random.randint(9, 16), random.randint(0, 59), 0))
        if balance > 0:
            add(dt, "Withdrawal", "Account Closure", "Branch", balance)
        elif balance < 0:
            add(dt, "Deposit", "Account Closure", "Branch", -balance)

    return rows


def monthly_rollup(acc_id, rows):
    """Aggregate one account's transactions into per-month statements."""
    months = {}
    prev_balance = 0
    for (dt, _a, _s, _t, direction, _c, _ch, amt, bal_after, _f) in rows:
        key = dt.strftime("%Y-%m")
        m = months.setdefault(key, {"open": prev_balance, "credits": 0, "debits": 0,
                                    "n": 0, "low": bal_after, "close": bal_after})
        if direction == "Credit":
            m["credits"] += amt
        else:
            m["debits"] += amt
        m["n"] += 1
        m["low"] = min(m["low"], bal_after)
        m["close"] = bal_after
        prev_balance = bal_after
    return [(acc_id, k, v["open"] / 100, v["close"] / 100, v["credits"] / 100,
             v["debits"] / 100, v["n"], v["low"] / 100) for k, v in months.items()]


## 8. Build the database

In [ ]:



def build_database():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)          # start fresh so the script is re-runnable

    branches = generate_branches()
    customers = generate_customers()
    accounts, holders = generate_account_specs(customers, branches)

    # Simulate every account, then give transactions chronological global IDs.
    all_txns, statements, final_balance = [], [], {}
    for acc in accounts:
        rows = simulate_account(acc)
        final_balance[acc["id"]] = rows[-1][8]
        all_txns.extend(rows)
        statements.extend(monthly_rollup(acc["id"], rows))
    all_txns.sort(key=lambda r: (r[0], r[1], r[2]))   # by datetime, then account, then seq

    con = sqlite3.connect(DB_PATH)
    con.executescript(SCHEMA)
    cur = con.cursor()

    cur.executemany("INSERT INTO branches VALUES (?,?,?,?,?,?,?,?)", branches)

    cur.executemany(
        "INSERT INTO customers VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
        [(c["id"], c["first"], c["last"], c["gender"], c["dob"].isoformat(),
          c["email"], c["phone"], c["city"], c["employment"], c["education"],
          float(c["income"]), c["score"], c["rating"], c["since"].isoformat())
         for c in customers])

    cur.executemany(
        "INSERT INTO accounts VALUES (?,?,?,?,?,?,?,?,?,?)",
        [(a["id"], a["branch_id"], a["number"], a["type"], a["status"],
          a["open"].isoformat(), a["close"].isoformat() if a["close"] else None,
          round(final_balance[a["id"]] / 100, 2), float(a["overdraft"]), a["rate"])
         for a in accounts])

    cur.executemany("INSERT INTO account_holders VALUES (?,?,?,?)", holders)

    cur.executemany(
        "INSERT INTO transactions VALUES (?,?,?,?,?,?,?,?,?,?)",
        [(i, r[1], r[0].strftime("%Y-%m-%d %H:%M:%S"), r[3], r[4], r[5], r[6],
          r[7] / 100, r[8] / 100, r[9])
         for i, r in enumerate(all_txns, start=1)])

    cur.executemany("INSERT INTO monthly_statements VALUES (?,?,?,?,?,?,?,?)", statements)

    con.commit()
    return con


## 9. Verification
Proves the requirements are met: row counts, integrity, foreign-key check and sample queries.

In [ ]:



def verify(con):
    cur = con.cursor()
    print("=" * 70)
    print("ROW COUNTS")
    for t in ["branches", "customers", "accounts", "account_holders",
              "transactions", "monthly_statements"]:
        n = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        cols = len(cur.execute(f"PRAGMA table_info({t})").fetchall())
        print(f"  {t:<20} {n:>8,} rows   {cols:>2} columns")

    print("\nINTEGRITY CHECKS")
    print("  PRAGMA integrity_check   :", cur.execute("PRAGMA integrity_check").fetchone()[0])
    fk = cur.execute("PRAGMA foreign_key_check").fetchall()
    print("  PRAGMA foreign_key_check :", "OK (no violations)" if not fk else fk[:5])
    bad = cur.execute("""
        SELECT COUNT(*) FROM accounts a
        WHERE ROUND(a.balance - (SELECT SUM(CASE direction WHEN 'Credit' THEN amount
                                                ELSE -amount END)
                                 FROM transactions t WHERE t.account_id = a.account_id), 2) <> 0
    """).fetchone()[0]
    print("  Accounts whose balance != sum of their transactions:", bad)
    no_primary = cur.execute("""
        SELECT COUNT(*) FROM accounts a WHERE NOT EXISTS
        (SELECT 1 FROM account_holders h WHERE h.account_id = a.account_id
         AND h.holder_role = 'Primary')""").fetchone()[0]
    print("  Accounts with no primary holder:", no_primary)

    print("\nSAMPLE: average income & credit score by ORDINAL credit rating")
    for r in cur.execute("""
        SELECT credit_rating, COUNT(*), ROUND(AVG(annual_income)), ROUND(AVG(credit_score))
        FROM customers GROUP BY credit_rating
        ORDER BY CASE credit_rating WHEN 'Poor' THEN 1 WHEN 'Fair' THEN 2 WHEN 'Good' THEN 3
                                    WHEN 'Very Good' THEN 4 ELSE 5 END"""):
        print(f"  {r[0]:<12} n={r[1]:>4}  avg income={r[2]:>8,.0f}  avg score={r[3]:.0f}")

    print("\nSAMPLE: spending by category (debits only, all accounts)")
    for r in cur.execute("""
        SELECT category, COUNT(*), ROUND(AVG(amount), 2)
        FROM transactions WHERE direction = 'Debit'
        GROUP BY category ORDER BY COUNT(*) DESC LIMIT 6"""):
        print(f"  {r[0]:<18} n={r[1]:>6,}  mean amount = {r[2]:>8,.2f}")

    print("\nSAMPLE JOIN across 4 tables (3 random-looking rows)")
    for r in cur.execute("""
        SELECT c.first_name || ' ' || c.last_name, b.branch_name, a.account_type,
               t.txn_datetime, t.txn_type, t.amount, t.balance_after
        FROM transactions t
        JOIN accounts a         ON a.account_id = t.account_id
        JOIN account_holders h  ON h.account_id = a.account_id AND h.holder_role = 'Primary'
        JOIN customers c        ON c.customer_id = h.customer_id
        JOIN branches b         ON b.branch_id = a.branch_id
        WHERE t.txn_id IN (100, 20000, 40000)"""):
        print("  ", r)
    print("=" * 70)


Builds `bank.db`

In [ ]:
connection = build_database()
verify(connection)
connection.close()
print(f"Created {DB_PATH} ({os.path.getsize(DB_PATH)/1_000_000:.1f} MB) with seed {SEED}.")

ROW COUNTS
  branches                   15 rows    8 columns
  customers               1,200 rows   14 columns
  accounts                1,810 rows   10 columns
  account_holders         1,988 rows    4 columns
  transactions           60,077 rows   10 columns
  monthly_statements     46,826 rows    8 columns

INTEGRITY CHECKS
  PRAGMA integrity_check   : ok
  PRAGMA foreign_key_check : OK (no violations)
  Accounts whose balance != sum of their transactions: 0
  Accounts with no primary holder: 0

SAMPLE: average income & credit score by ORDINAL credit rating
  Poor         n=  55  avg income=   6,833  avg score=556
  Fair         n= 435  avg income=  19,121  avg score=633
  Good         n= 496  avg income=  33,497  avg score=703
  Very Good    n= 194  avg income=  40,763  avg score=763
  Exceptional  n=  20  avg income=  47,860  avg score=814

SAMPLE: spending by category (debits only, all accounts)
  Transfer           n= 5,997  mean amount =   194.08
  Groceries          n= 5,880  

## Explore the data

In [ ]:
import pandas as pd
con = sqlite3.connect(DB_PATH)
pd.read_sql_query("SELECT * FROM customers LIMIT 5", con)

,customer_id,first_name,last_name,gender,date_of_birth,email,phone,city,employment_status,education_level,annual_income,credit_score,credit_rating,customer_since
0,1,Charlotte,Lewis,Female,1995-10-03,charlotte.lewis1@example.com,07700 900067,London,Employed,Master,14000.0,683,Good,2019-10-05
1,2,James,Ali,Male,1979-11-30,james.ali2@example.com,07700 900271,London,Employed,Bachelor,50300.0,678,Good,2006-05-29
2,3,Ethan,Price,Male,1970-07-10,ethan.price3@example.com,07700 900093,Nottingham,Student,Master,1900.0,601,Fair,2022-02-19
3,4,Sophie,Price,Female,1994-12-30,sophie.price4@example.com,07700 900881,Swansea,Student,Bachelor,7400.0,591,Fair,2025-05-28
4,5,Rebecca,Baker,Female,1981-06-02,rebecca.baker5@example.com,07700 900995,Liverpool,Employed,Secondary,20600.0,572,Poor,2011-10-25


In [ ]:
pd.read_sql_query("""
SELECT c.first_name, c.last_name, a.account_type, b.branch_name, t.txn_datetime, t.txn_type, t.amount, t.balance_after
FROM transactions t
JOIN accounts a        ON a.account_id = t.account_id
JOIN account_holders h ON h.account_id = a.account_id AND h.holder_role = 'Primary'
JOIN customers c       ON c.customer_id = h.customer_id
JOIN branches b        ON b.branch_id = a.branch_id
LIMIT 10
""", con)

,first_name,last_name,account_type,branch_name,txn_datetime,txn_type,amount,balance_after
0,Charlotte,Lewis,Current,London Central,2019-10-05 09:36:55,Deposit,765.80,765.80
1,Charlotte,Lewis,Current,London Central,2019-10-06 15:04:34,Withdrawal,20.00,745.80
2,Charlotte,Lewis,Current,London Central,2019-11-08 11:27:51,Card Payment,10.63,735.17
3,Charlotte,Lewis,Current,London Central,2020-01-05 18:55:54,Card Payment,13.19,721.98
4,Charlotte,Lewis,Current,London Central,2020-07-24 15:48:48,Card Payment,22.84,699.14
5,Charlotte,Lewis,Current,London Central,2020-08-31 07:48:13,Card Payment,68.77,630.37
6,Charlotte,Lewis,Current,London Central,2020-11-01 12:42:18,Direct Debit,59.81,570.56
7,Charlotte,Lewis,Current,London Central,2021-03-21 12:55:43,Salary,893.58,1464.14
8,Charlotte,Lewis,Current,London Central,2021-10-15 10:44:13,Card Payment,39.60,1424.54
9,Charlotte,Lewis,Current,London Central,2022-01-07 06:48:37,Card Payment,20.95,1403.59


## Download the database

In [ ]:
try:
    from google.colab import files
    files.download(DB_PATH)
except ImportError:
    print(f'Not running on Colab - the file is at {os.path.abspath(DB_PATH)}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>